# satelliteagent-env-prep

Lightweight prep notebook that ONLY builds the SatelliteAgent wheel
(`satellite-agent`). Kept separate from the main `prime-rl-offline-prep`
because the main prep is heavy (~15-20 min, 277 wheels + 2 GB of models)
while SatelliteAgent itself changes frequently as Phase 2 progresses, so we
want a quick rebuild loop here (~1-2 min).

Output (`/kaggle/working/output/wheels/satellite_agent-*.whl`) is mounted
alongside the main prep output via `kernel_sources` in S5+ notebooks.

Internet ON, CPU only.

In [ ]:
import os, subprocess

SA_REPO = "https://github.com/masato-todo/SatelliteAgent.git"
SA_BRANCH = "Branch/refactor"
SA_DIR = "/kaggle/working/satelliteagent"
OUT_DIR = "/kaggle/working/output"
OUT_WHEELS = f"{OUT_DIR}/wheels"
os.makedirs(OUT_WHEELS, exist_ok=True)

subprocess.run(f"rm -rf {SA_DIR}", shell=True, check=True)
subprocess.run(
    f"git clone --depth 1 --branch {SA_BRANCH} {SA_REPO} {SA_DIR}",
    shell=True, check=True,
)
subprocess.run(f"git -C {SA_DIR} log -1 --oneline", shell=True)
print()
subprocess.run(f"ls {SA_DIR}", shell=True)
print()
subprocess.run(f"ls {SA_DIR}/satelliteagent_env 2>&1 || echo 'satelliteagent_env dir missing'", shell=True)

# Record the exact commit we built from so downstream notebooks
# (S12+) have a stable provenance trail. The file lands inside
# /kaggle/working/output/, which is what S* notebooks consume via
# `kernel_sources`.
sha = subprocess.check_output(f"git -C {SA_DIR} rev-parse HEAD", shell=True).decode().strip()
short_sha = subprocess.check_output(f"git -C {SA_DIR} rev-parse --short HEAD", shell=True).decode().strip()
oneline = subprocess.check_output(f"git -C {SA_DIR} log -1 --pretty=format:'%h %s (%ai)'", shell=True).decode().strip()
provenance = (
    f"repo:    {SA_REPO}\n"
    f"branch:  {SA_BRANCH}\n"
    f"sha:     {sha}\n"
    f"short:   {short_sha}\n"
    f"head:    {oneline}\n"
)
with open(f"{OUT_DIR}/sa_commit.txt", "w") as f:
    f.write(provenance)
print("\n=== provenance ===")
print(provenance)

In [ ]:
import os, subprocess
SA_DIR = "/kaggle/working/satelliteagent"
OUT_WHEELS = "/kaggle/working/output/wheels"

# Build wheel. --no-deps: deps are resolved by the offline notebook from
# the main prep wheels.
subprocess.run(
    f"python3.12 -m pip wheel {SA_DIR} --no-deps -w {OUT_WHEELS}",
    shell=True, check=True,
)

print("--- built wheels ---")
subprocess.run(f"ls -la {OUT_WHEELS}", shell=True)
print()
subprocess.run(f"ls {OUT_WHEELS} | grep -E '^satellite' || echo NO_SA_WHEEL", shell=True)

In [ ]:
# Sanity: install the wheel and import the env package end-to-end
import subprocess, glob
wheels = glob.glob("/kaggle/working/output/wheels/satellite_agent-*.whl")
if not wheels:
    raise RuntimeError("satellite-agent wheel not built")
subprocess.run(f"python3.12 -m pip install --force-reinstall --no-deps {wheels[0]}", shell=True, check=True)
subprocess.run(
    "python3.12 -c 'import satelliteagent_env; print(\"satelliteagent_env imports OK,\", satelliteagent_env.load_environment)'",
    shell=True, check=True,
)